[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nlavadodatasci/atd-megaf-fiabilidade/blob/main/T3_escolha_da_distribuicao/T3_escolha_da_distribuicao.ipynb)

# T3 — Escolha da distribuição

**ATD — MEGAF 2026/27 · Atividade de Fiabilidade · Tema T3**

O *notebook* comum (`00_comum/testes_de_ajustamento.ipynb`) mostra como testar se uma distribuição serve. Este acrescenta duas coisas: um critério para **comparar** distribuições entre si (AIC) e uma forma de ver **que diferença faz** escolher mal. Faça primeiro o *notebook* comum.

**Como usar:** execute as células por ordem (Shift+Enter). As primeiras secções estão resolvidas, com dados da aula ou com exemplos inventados, para mostrar o método. A última secção, **A sua vez**, lê os dados da instalação e deixa pistas: o trabalho do grupo começa aí. O capítulo deste tema no documento de apoio explica cada passo.

## 1. Ler os dados da aula e ajustar as quatro distribuições

In [1]:
import os, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

REPO = "https://raw.githubusercontent.com/nlavadodatasci/atd-megaf-fiabilidade/main/dados/"

def obter(ficheiro):
    """Vai buscar o ficheiro ao repositório. Se falhar, abre uma janela
    para o escolher no seu computador."""
    if not os.path.exists(ficheiro):
        try:
            urllib.request.urlretrieve(REPO + ficheiro, ficheiro)
        except Exception:
            from google.colab import files
            print("Escolha no seu computador o ficheiro", ficheiro)
            carregados = files.upload()
            if ficheiro not in carregados:               # veio com outro nome: renomeia
                os.rename(list(carregados)[0], ficheiro)
    return ficheiro

aula = pd.read_excel(obter("tempo_falhas_aula.xlsx"), sheet_name="tempo_falhas", usecols="A:B")
t = aula.iloc[:, 1].dropna().to_numpy()      # tempos até à falha da aula (horas)
print("Número de tempos:", len(t), "  MTBF:", round(t.mean(), 2))

def ajustar_modelos(tempos):
    # devolve, para cada distribuição, o modelo ajustado e o n.º de parâmetros estimados
    media, desvio = tempos.mean(), tempos.std(ddof=1)
    logs = np.log(tempos)
    beta, _, eta = stats.weibull_min.fit(tempos, floc=0)
    return {"Exponencial": (stats.expon(scale=media), 1),
            "Normal":      (stats.norm(media, desvio), 2),
            "Lognormal":   (stats.lognorm(s=logs.std(ddof=1), scale=np.exp(logs.mean())), 2),
            "Weibull":     (stats.weibull_min(beta, scale=eta), 2)}

modelos = ajustar_modelos(t)

Número de tempos: 9998   MTBF: 8738.14


## 2. Comparar modelos: verosimilhança e AIC

Um teste de ajustamento responde a "este modelo é aceitável?". Com poucos dados, é frequente vários modelos serem aceitáveis ao mesmo tempo, e é preciso outra forma de escolher.

A **log-verosimilhança** mede quão prováveis são os dados segundo o modelo: é a soma de $\ln f(t_i)$ para todos os tempos observados. Quanto maior, melhor o ajuste. Em `scipy`: `modelo.logpdf(tempos).sum()`.

Um modelo com mais parâmetros ajusta-se sempre um pouco melhor, mesmo sem razão para isso. O critério de Akaike desconta esse efeito:

$$\text{AIC} = 2m - 2\ln L$$

em que $m$ é o número de parâmetros estimados. **Prefere-se o modelo com menor AIC.** Só as diferenças contam: uma diferença inferior a 2 não distingue dois modelos.

In [2]:
def tabela_aic(tempos):
    linhas = []
    for nome, (modelo, m) in ajustar_modelos(tempos).items():
        logL = modelo.logpdf(tempos).sum()
        linhas.append({"distribuição": nome, "m": m, "logL": logL, "AIC": 2 * m - 2 * logL})
    tab = pd.DataFrame(linhas).set_index("distribuição")
    tab["diferença para o melhor"] = tab["AIC"] - tab["AIC"].min()
    return tab.round(1)

tabela_aic(t)

,m,logL,AIC,diferença para o melhor
distribuição,,,,
Exponencial,1,-100734.4,201470.7,0.0
Normal,2,-104902.8,209809.5,8338.8
Lognormal,2,-101713.1,203430.3,1959.5
Weibull,2,-100734.3,201472.5,1.8


Nos dados da aula, a exponencial e a Weibull ficam praticamente empatadas (a Weibull ajustada tem $\beta \approx 1$, ou seja, é quase uma exponencial) e a exponencial ganha por ter um parâmetro a menos. A normal e a lognormal ficam muito atrás.

## 3. Que diferença faz escolher mal?

A pergunta prática é: se usar o modelo errado, a fiabilidade prevista sai muito diferente? Com tantos dados como os da aula, podemos comparar cada modelo com a proporção de unidades que de facto sobreviveram.

In [3]:
instantes = [1000, 5000, 20000]
tab = pd.DataFrame({"empírica": [(t > x).mean() for x in instantes]}, index=instantes)
for nome, (modelo, m) in modelos.items():
    tab[nome] = modelo.sf(instantes)
tab.index.name = "t (horas)"
tab.round(3)

,empírica,Exponencial,Normal,Lognormal,Weibull
t (horas),,,,,
1000,0.890,0.892,0.813,0.888,0.891
5000,0.566,0.564,0.666,0.492,0.563
20000,0.101,0.101,0.098,0.139,0.102


O erro de um modelo mal escolhido não é igual em todos os instantes: pode ser pequeno a meio da vida e grande no princípio ou no fim. Para a instalação interessa o que acontece no tempo de missão.

## 4. A sua vez

Pistas:

1. Para cada um dos cinco componentes, aplique os testes do *notebook* comum (função `comparar`) e a tabela do AIC. Olhe também para os gráficos quantil–quantil.
2. Com 50 unidades, é provável que mais do que uma distribuição não seja rejeitada. Como desempata? O AIC e o gráfico QQ concordam?
3. Há alguma razão física para preferir uma distribuição (secção 4 do documento de apoio)?
4. Calcule $R(2000)$ com a melhor e com a pior distribuição candidata de cada componente. Em que componentes a escolha faz mais diferença?
5. Um valor de $R(2000)$ pode ser absurdo (por exemplo, por o modelo admitir tempos negativos). Repare se isso acontece.

In [4]:
ensaios = pd.read_excel(obter("dados_fiabilidade.xlsx"), sheet_name="ensaios_vida")

for componente in ensaios["componente"].unique():
    tc = ensaios[ensaios["componente"] == componente]["tempo_ate_falha_h"].to_numpy()
    print(componente, "- número de tempos:", len(tc))
    # display(tabela_aic(tc))

Interruptor - número de tempos: 50
Transformador - número de tempos: 50
QGBT - número de tempos: 50
Gerador - número de tempos: 50
Controlador - número de tempos: 50
